In [ ]:
# Imports and setup
import os
os.chdir("/home/ubuntu/rhardy-us-east-1/code/coloncrafter/")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from joblib import Parallel, delayed
from src.data.c3vd import C3VDDataset
from src.style import StyleTransferPipeline2D
from tqdm.auto import tqdm
from PIL import Image

In [ ]:
# Load the dataset
root_dir = "/home/ubuntu/rhardy-us-east-1/code/coloncrafter/example_data/c3vd"
section = "cecum_t1_a"
resize = (512, 512)
depth_scale = 655.35

dataset = C3VDDataset(
    root_dir=root_dir, 
    section=section,
    resize=resize,
    depth_scale=depth_scale,
)
print(f"Number of frames: {len(dataset)}")

def load_image(idx):
    return dataset[idx]["image"]

images = Parallel(n_jobs=-1)(
    delayed(load_image)(i) for i in tqdm(range(len(dataset)))
)
video = torch.stack(images, dim=0)

In [ ]:
# Load the style image
style_path = "/home/ubuntu/rhardy-us-east-1/code/coloncrafter/example_data/style/style.png"
style_image = np.array(Image.open(style_path))[..., : 3]
style_image = torch.from_numpy(style_image).float() / 255.0
style_image = 2.0 * style_image - 1.0
style_image = style_image.permute(2, 0, 1)

num_frames, c, h, w = video.shape
style = F.interpolate(style_image.unsqueeze(0), size=(h, w), mode="bilinear", align_corners=False)
style = style.repeat(num_frames, 1, 1, 1)

In [ ]:
# Load the style transfer pipeline
pipeline = StyleTransferPipeline2D(
    model_id="CompVis/stable-diffusion-v1-4",
    options={"gamma": 0.75, "tau": 1.5},
    device="cuda",
    dtype=torch.float16,
)

In [ ]:
# Run style transfer
num_inference_steps = 25
partial_inversion_fraction = 0.4
alpha = 0.0
inpaint_latents = True
match_histograms = True
window_size = 8
overlap = 0

output = np.zeros((num_frames, h, w, c))
counts = np.zeros((num_frames, h, w, c))

with torch.inference_mode():
    for i in tqdm(range(0, video.shape[0] - window_size + 1, window_size - overlap)):
        images = pipeline.run(
            video[i : i + window_size].to(pipeline.device, pipeline.dtype), 
            style[i : i + window_size].to(pipeline.device, pipeline.dtype),
            num_inference_steps=num_inference_steps,
            partial_inversion_fraction=partial_inversion_fraction,
            alpha=alpha,
            inpaint_latents=inpaint_latents,
            match_histograms=match_histograms,
        )
        output[i : i + window_size] += images
        counts[i : i + window_size] += 1
        torch.cuda.empty_cache()

    if i + window_size < num_frames:
        images = pipeline.run(
            video[i + window_size :].to(pipeline.device, pipeline.dtype), 
            style[i + window_size :].to(pipeline.device, pipeline.dtype),
            num_inference_steps=num_inference_steps,
            partial_inversion_fraction=partial_inversion_fraction,
            alpha=alpha,
            inpaint_latents=inpaint_latents,
            match_histograms=match_histograms,
        )
        output[i + window_size :] += images
        counts[i + window_size :] += 1
        torch.cuda.empty_cache()

output = output / counts

In [ ]:
# Plot the results
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
axs[0].imshow(video[0].permute(1, 2, 0).cpu().numpy().astype(np.float32) * 0.5 + 0.5)  
axs[1].imshow(video[0].mean(dim=0)[..., None].repeat(1, 1, 3).cpu().numpy().astype(np.float32) * 0.5 + 0.5)
axs[2].imshow(output[0])
axs[3].imshow(np.abs(video[0].mean(dim=0)[..., None].repeat(1, 1, 3).cpu().numpy().astype(np.float32) * 0.5 + 0.5 - output[0]))
axs[0].set_axis_off()
axs[1].set_axis_off()
axs[2].set_axis_off()
axs[3].set_axis_off()
plt.tight_layout()
plt.show()